# UC CosMx: Validation 1K Processing and Annotation Comparison (Supp 6L)

Serial-cut 1K validation experiment:
1. QC and normalization
2. AUCell labels before and after GCN imputation
3. Marker dot plots before and after imputation (Supp 6L)

In [ ]:
import scanpy as sc
import pandas as pd
from scipy.sparse import csr_matrix

In [ ]:
sample_dir = '/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/flat'

In [ ]:
counts = pd.read_csv(sample_dir+'/UC_val_dir_exprMat_file.csv', index_col=0)          # Rows = genes, Cols = cells


In [ ]:
fov_positions = pd.read_csv(sample_dir+'/UC_val_dir_fov_positions_file.csv')  # Contains x, y


In [ ]:
metadata = pd.read_csv(sample_dir+'/UC_val_dir_metadata_file.csv')      # Indexed by cell_id

In [ ]:
counts_clean = counts.drop(['cell_ID'],axis = 1)

In [ ]:
counts_clean.index = counts_clean['cell']
counts_clean = counts_clean.drop(['cell'],axis = 1)

In [ ]:
# Create AnnData object
adata = sc.AnnData(X=counts_clean)

In [ ]:
# Add metadata to AnnData
adata.obs = metadata

In [ ]:
adata.var['gene'] = adata.var_names.tolist()

In [ ]:
adata.obsm['spatial_fov']=adata.obs[['CenterX_global_px', 'CenterY_global_px']].values.astype(int)

In [ ]:
adata.var['neg_probe']=[True if 'Negative' in i else False for i in adata.var.index.tolist()]
adata.var['control_probe']=[True if 'SystemControl' in i else False for i in adata.var.index.tolist()]

In [ ]:
adata=adata[:, ~adata.var["neg_probe"]]
adata=adata[:, ~adata.var["control_probe"]]

In [ ]:

if not isinstance(adata.X, csr_matrix):
    adata.X = csr_matrix(adata.X)


In [ ]:

sc.pp.calculate_qc_metrics(
    adata, inplace=True, log1p=True
)

In [ ]:
# only remove cells that have too little gene counts
sc.pp.filter_cells(adata, min_counts=50)
sc.pp.filter_cells(adata, min_genes=25)
sc.pp.filter_cells(adata, max_counts=800)
sc.pp.filter_cells(adata, max_genes=350)
adata=adata[adata.obs['Area']<=30000]

In [ ]:
adata.obs[['orig.ident']]='UC_V2_UC_V1'
adata.obs['patient']=['UC_V1' if int(i) <=28 else 'UC_V2' for i in adata.obs['fov'].tolist() ]
adata.obs['fov_cell_id']= [str(i)+'_'+str(j) for i,j in zip(adata.obs['fov'].tolist(), adata.obs['cell_ID'].tolist())]


In [ ]:
slide=[]
for i,j in zip(adata.obs['orig.ident'].tolist(),adata.obs['fov'].tolist()):
    j= int(j)
    if j <=12:
        slide.append(i+'_'+'slide1')
    elif j > 12 and j <= 28:
        slide.append(i+'_'+'slide2')
    elif j > 28 and j <= 39:
        slide.append(i+'_'+'slide3')
    else:
        slide.append(i+'_'+'slide4')

In [ ]:
adata.obs['slide']=slide

In [ ]:
adata.layers['raw']=pd.DataFrame(adata.X.toarray())

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)


In [ ]:
adata.write_h5ad('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/h5ad_obj/qc_norm_log1p.h5ad')


In [ ]:
adata_norm_nolog1p = adata.copy()
adata_norm_nolog1p.X = adata.layers['raw']

In [ ]:
sc.pp.normalize_total(adata_norm_nolog1p, inplace=True)

In [ ]:
adata_norm_nolog1p_count = pd.DataFrame(adata_norm_nolog1p.X)

In [ ]:
adata_norm_nolog1p_count.columns = adata_norm_nolog1p.var.index.tolist()
adata_norm_nolog1p_count.index=adata_norm_nolog1p.obs['fov_cell_id'].tolist()

In [ ]:
adata_norm_nolog1p_count.to_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/count_csv/norm_counts.csv')

# anno

In [ ]:
adata=sc.read_h5ad('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/h5ad_obj/qc_norm_log1p.h5ad')


In [ ]:
noimp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir/anno/norm_counts_aucell.csv')

In [ ]:
imp_anno = pd.read_csv('/path/to/cosmx_data/uc_serial_cuts/1000plex/UC_val_dir//anno/imp_log1p_counts_aucell.csv')

In [ ]:
mapper= pd.read_csv('/path/to/scrna/uc/cell_type_name_mapper.csv')

In [ ]:
imp_anno=imp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')
noimp_anno=noimp_anno.merge(mapper, left_on = 'label',right_on='cell_type_short',how='left')

In [ ]:
adata.obs['v7_ywl_b_imp_cell_type_short'] = imp_anno['cell_type_short'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_full'] = imp_anno['cell_type'].tolist()
adata.obs['v7_ywl_b_imp_cell_category_from_type'] = imp_anno['cell_category'].tolist()
adata.obs['v7_ywl_b_imp_cell_type_general'] = imp_anno['cell_type_general'].tolist()

In [ ]:
adata.obs['cell_type_short'] = noimp_anno['cell_type_short'].tolist()
adata.obs['cell_type_full'] = noimp_anno['cell_type'].tolist()
adata.obs['cell_category_from_type'] = noimp_anno['cell_category'].tolist()
adata.obs['cell_type_general'] = noimp_anno['cell_type_general'].tolist()

# Marker dot plots before and after imputation (Supp 6L)

In [ ]:
markers = { 'Connective': [ 'COL1A1', 'TAGLN', 'COL3A1','IGFBP5'], 
            'Epithelial': [ 'KRT8', 'EPCAM', 'KRT19', 'CLDN4'],
           'Myeloid': [ 'LYZ','C1QB','C1QA', 'MS4A6A', 'CSF1R'], 
                     'Plasma': ['JCHAIN','IGHG1','IGKC'],
           'T': ['CD3E','CD8B','CCL5','SPOCK2','CD28']}

In [ ]:
adata_key = adata[adata.obs['cell_category_from_type'].isin(['T','Myeloid','Epithelial','Connective','Plasma'])]
dp = sc.pl.dotplot(
    adata_key, markers, 'cell_category_from_type', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()


In [ ]:
adata_key = adata[adata.obs['v7_ywl_b_imp_cell_category_from_type'].isin(['T','Myeloid','Epithelial','Connective','Plasma'])]
dp = sc.pl.dotplot(
    adata_key, markers, 'v7_ywl_b_imp_cell_category_from_type', 
    return_fig=True, 
    cmap='magma', 
    vmin=0, 
    vmax=0.4
)
dp.add_totals().style(
    cmap='magma', 
    dot_edge_color='black', 
    dot_edge_lw=0.5, 
    dot_min=0,  # Set minimum dot size
    dot_max=0.5   # Set maximum dot size
).show()
